In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

RAVLT_CUTOFF = 12  # RAVLT delayed recall superager threshold

# Paths
EXPORT_DIR = Path("/Users/rachelmorse/Documents/2023:2024/Data/Exported data")
NORMS_DIR = Path("/Users/rachelmorse/superagers/superager_classification")
BBHI_FILE = EXPORT_DIR / "clean_bbhi_tp1.csv"
BBHI_SENIOR_FILE = EXPORT_DIR / "clean_bbhi_senior_tp1.csv"
NORMS_AGE_FILE = NORMS_DIR / "neuronorma_neuropsych_data.xlsx"
NORMS_EDUCATION_FILE = NORMS_DIR / "neuronorma_education_data.xlsx"

In [ ]:
# Read in clean data
bbhi_senior = pd.read_csv(BBHI_SENIOR_FILE)
bbhi = pd.read_csv(BBHI_FILE)

# Align shared column types (skip id and sex)
ignore = {"id", "sex"}
shared_columns = [c for c in bbhi_senior.columns if c in bbhi.columns and c not in ignore]
for col in shared_columns:
    bbhi_senior[col] = pd.to_numeric(bbhi_senior[col], errors="coerce")
    bbhi[col] = pd.to_numeric(bbhi[col], errors="coerce")

print(f"Number of BBHI senior participants: {len(bbhi_senior)}")
print(f"Number of BBHI participants: {len(bbhi)}")

# Merge datasets
data = pd.concat([bbhi, bbhi_senior], ignore_index=True, sort=False)

# Require tp2 ages only when they exist
data = data[data["w1_age"].notna()]

# Print the number of participants after filtering
print(f"Number of participants: {len(data)}")

data.sample(5)

In [ ]:
def print_age_summary(df: pd.DataFrame) -> None:
    """Print the number of participants and the mean, SD, and range of baseline age.

    Args:
        df: Participant data with a w1_age column.
    """
    ages = df["w1_age"]
    print(f"Number of participants: {ages.count()}")
    print(f"Mean age: {ages.mean():.2f}")
    print(f"SD age: {ages.std():.2f}")
    print(f"Range age: {ages.min():.2f} - {ages.max():.2f}")


# Look at data for available cross-sectional analysis
print_age_summary(data)

In [ ]:
# Read in the Neuronorma data (from the two publications above) in excel form & put into a df
xls = pd.ExcelFile(NORMS_AGE_FILE)
score_mappings = {sheet_name: pd.read_excel(xls, sheet_name) for sheet_name in xls.sheet_names}

# Pre-compute supported age bounds for clipping
age_bounds = []
for sheet_name in score_mappings:
    match = re.split(r"[–-]", sheet_name)
    try:
        low, high = map(int, match[:2])
        age_bounds.append((low, high))
    except Exception:
        continue

if age_bounds:
    MIN_SUPPORTED_AGE = min(b[0] for b in age_bounds)
    MAX_SUPPORTED_AGE = max(b[1] for b in age_bounds)
else:
    MIN_SUPPORTED_AGE = None
    MAX_SUPPORTED_AGE = None

In [ ]:
# Round down years of education data because some participants have data that is not a whole number (e.g. YoE = 9.5)
# This gives participants the lower education value because they will have their cognitive test scores normalized according to education level, disadvantaging them if it is rounded up
data["YoE"] = np.floor(data["YoE"])

# Round the ages of the participants to the nearest whole number because you need a whole number for calculating the norms
data["w1_age_round"] = np.round(data["w1_age"])
data["w2_age_round"] = np.round(data["w2_age"])

In [ ]:
def raw_score_matches(raw_score: float, raw_score_range: str) -> bool:
    """Check whether a raw score falls in a raw score range from the Neuronorma tables.

    Args:
        raw_score: Raw neuropsych score for the participant
        raw_score_range: Raw score range from the table (a single value, a range, or ≥ or ≤ a value)

    Returns:
        True if the raw score is in the range, False otherwise or if there is no data
    """
    if raw_score_range in {"-", "—"}:
        return False  # skip this row if there is no data
    if "≥" in raw_score_range:
        return raw_score >= int(raw_score_range.replace("≥", ""))
    if "≤" in raw_score_range:
        return raw_score <= int(raw_score_range.replace("≤", ""))
    if "–" in raw_score_range:
        min_raw_score, max_raw_score = map(int, raw_score_range.split("–"))
        return min_raw_score <= raw_score <= max_raw_score
    # raw_score_range is a single value
    return raw_score == int(raw_score_range)


# Function to create scaled scores for that adjust for age
def map_raw_to_scaled_age(raw_score: float, age: float, var: str) -> float:
    """Maps raw scores to scaled scores based on age.

    Args:
        raw_score: Raw neuropsych score for the participant
        age: Age of the participant
        var: Variable name for the raw score in the score_mappings DataFrame

    Returns:
        Scaled score corresponding to the raw score and age, or NaN if not found
    """
    if pd.isna(raw_score) or pd.isna(age):
        return np.nan

    age = int(np.round(age))
    if MIN_SUPPORTED_AGE is not None and MAX_SUPPORTED_AGE is not None:
        age = int(np.clip(age, MIN_SUPPORTED_AGE, MAX_SUPPORTED_AGE))

    # Find the appropriate score mapping based on the age
    for age_range, score_mapping in score_mappings.items():
        try:
            min_age, max_age = map(int, re.split(r"[–-]", age_range)[:2])
        except Exception:
            continue
        if min_age <= age <= max_age:
            for _, row in score_mapping.iterrows():
                if raw_score_matches(raw_score, str(row[var])):
                    return row["Scaled Score"]
    return np.nan

In [ ]:
def map_raw_to_scaled_edu(
    data: pd.DataFrame, edu_table: pd.DataFrame, norm_age_var: str, out_col: str = "norm_score", max_yoe: int = 20
) -> pd.DataFrame:
    """Adds scaled scores from a Neuronorma table adjusted for education.

    Args:
        data: The neuropsych dataset.
        edu_table: DataFrame containing the normative lookup table.
        norm_age_var: Column in neuropsych dataset with normative age index.
        out_col: Name of output column for the scaled scores.
        max_yoe: Cap for years of education in the lookup table.

    Returns:
        The neuropsych dataset with an added column for scaled scores.
    """

    def lookup(row: pd.Series) -> float:
        try:
            age_idx = int(row[norm_age_var])
            yoe = int(min(row["YoE"], max_yoe))
            age_idx = max(min(age_idx, edu_table.index.max()), edu_table.index.min())
            min_yoe = min(edu_table.columns)
            max_yoe_col = max(edu_table.columns)
            yoe = max(min(yoe, max_yoe_col), min_yoe)
            if yoe not in edu_table.columns:
                yoe = max(c for c in edu_table.columns if c <= yoe)
            return edu_table.loc[age_idx, yoe]
        except Exception:
            return np.nan

    data[out_col] = data.apply(lookup, axis=1)
    return data


def add_scaled_scores(data: pd.DataFrame, sheet: str, raw_stem: str, norm_stem: str) -> pd.DataFrame:
    """Adds scaled scores that adjust for age and education for one test at tp1 and tp2.

    Args:
        data: The neuropsych dataset.
        sheet: Neuronorma sheet name for the test (e.g. "TMTB").
        raw_stem: Raw score column without the timepoint prefix (e.g. "tmt_b_raw").
        norm_stem: Start of the new scaled score column names (e.g. "tmtb").

    Returns:
        The neuropsych dataset with added age-adjusted and age- and education-adjusted scaled score columns.
    """
    for tp in ["w1", "w2"]:
        norm_age_var = f"{tp}_{norm_stem}_norm_age"
        data[norm_age_var] = [
            map_raw_to_scaled_age(raw_score, age, sheet)
            for raw_score, age in zip(data[f"{tp}_{raw_stem}"], data[f"{tp}_age_round"])
        ]
        data = map_raw_to_scaled_edu(
            data, edu_table=edu_tables[sheet], norm_age_var=norm_age_var, out_col=f"{tp}_{norm_stem}_norm"
        )
    return data


# Load education adjustment tables
edu_tables = {
    sheet: pd.read_excel(NORMS_EDUCATION_FILE, sheet_name=sheet, index_col=0) for sheet in ["TMTB", "DS_B", "SF"]
}

In [ ]:
# Create scaled scores for TMT-B that adjust for age and education at tp1 and tp2
data = add_scaled_scores(data, "TMTB", "tmt_b_raw", "tmtb")

data[
    ["id", "YoE", "w1_age_round", "w1_tmt_b_raw", "w1_tmtb_norm", "w2_age_round", "w2_tmt_b_raw", "w2_tmtb_norm"]
].sample(10)

In [ ]:
# Create scaled scores for inverse digits that adjust for age and education at tp1 and tp2
data = add_scaled_scores(data, "DS_B", "inverse_digits_raw", "dsb")

data[
    [
        "id",
        "YoE",
        "w1_age_round",
        "w1_inverse_digits_raw",
        "w1_dsb_norm",
        "w2_age_round",
        "w2_inverse_digits_raw",
        "w2_dsb_norm",
    ]
].sample(10)

In [ ]:
# Create scaled scores for semantic fluency that adjust for age and education at tp1 and tp2
data = add_scaled_scores(data, "SF", "sem_fluency_raw", "sf")

data[
    [
        "id",
        "YoE",
        "w1_age_round",
        "w1_sem_fluency_raw",
        "w1_sf_norm",
        "w2_age_round",
        "w2_sem_fluency_raw",
        "w2_sf_norm",
    ]
].sample(10)

In [ ]:
# Calculate who is superager based off of RAVLT score

# Schmidt 1996 - age 16-29 RAVLT-Delayed recall is scoring 12+ (no data adjusted by sex)
data.loc[data["w1_delayed_recall_raw"] >= RAVLT_CUTOFF, "w1_superager_ravlt"] = 1
data.loc[data["w1_delayed_recall_raw"] < RAVLT_CUTOFF, "w1_superager_ravlt"] = 0

# Repeat at tp2
data.loc[data["w2_delayed_recall_raw"] >= RAVLT_CUTOFF, "w2_superager_ravlt"] = 1
data.loc[data["w2_delayed_recall_raw"] < RAVLT_CUTOFF, "w2_superager_ravlt"] = 0

# Look at mean and standard deviation of RAVLT scores
mean_ravlt = data["w1_delayed_recall_raw"].mean()
print(f"Mean RAVLT score at tp1: {mean_ravlt:.2f}")

# Print mean age
mean_age = data["w1_age"].mean()
print(f"Mean age of participants at tp1: {mean_age:.2f}")

print("Number of superagers by RAVLT criteria at tp1:")
print(data[data["w1_superager_ravlt"] == 1]["id"].count())

print("Number of superagers by RAVLT criteria at tp2:")
print(data[data["w2_superager_ravlt"] == 1]["id"].count())

# Manually check that everything is running correctly
data[["id", "w1_delayed_recall_raw", "w1_superager_ravlt", "w2_superager_ravlt"]].head(10)

With the scaled scores calculated, the mean is 10 and the SD is 3 for all variables. For the Neuronorma data, see [Peña-Casanova et al. (2009b)](https://pubmed.ncbi.nlm.nih.gov/19549723/) for more info. 

In [ ]:
# Create a superager variable that = 1 when superagers are above 1SD below the norm for TMT-B, semantic fluency, and inverse digits and meet the RAVLT criteria
def meets_nonmemory_threshold(row: pd.Series, prefix: str) -> bool:
    """Check if a participant meets non-episodic memory superager thresholds.

    Args:
        row: A row from the DataFrame.
        prefix: The prefix for the timepoint (e.g., w1).

    Returns:
        True if the participant meets the thresholds, False otherwise.
    """
    cols = [f"{prefix}_tmtb_norm", f"{prefix}_sf_norm", f"{prefix}_dsb_norm"]
    values = row[cols]
    if values.isna().any():
        return False
    lower_bound = 10 - 1 * 3  # scaled score of 10 is the mean and SD is 3 for all variables
    return (values >= lower_bound).all()


def qualifies(row: pd.Series, prefix: str) -> float:
    """Check if a participant qualifies as a superager at one timepoint.

    Args:
        row: A row from the DataFrame.
        prefix: The prefix for the timepoint (e.g., w1).

    Returns:
        1 if the participant qualifies as a superager, 0 otherwise, NaN if insufficient data.
    """
    required_cols = [f"{prefix}_superager_ravlt", f"{prefix}_tmtb_norm", f"{prefix}_sf_norm", f"{prefix}_dsb_norm"]
    if row[required_cols].isna().any():
        return np.nan
    return int(row[f"{prefix}_superager_ravlt"] == 1 and meets_nonmemory_threshold(row, prefix))


data["superager_tp1"] = data.apply(qualifies, axis=1, prefix="w1")
data["superager_tp2"] = data.apply(qualifies, axis=1, prefix="w2")
longitudinal_mask = ~data[["superager_tp1", "superager_tp2"]].isna().any(axis=1)
data["superager_long"] = np.where(
    longitudinal_mask, ((data["superager_tp1"] == 1) & (data["superager_tp2"] == 1)).astype(int), np.nan
)

# Display a random sample of 10 rows
sample_df = data.sample(10)
print(data[data["superager_long"] == 1]["id"].count())

# Display the relevant columns
relevant_columns = [
    "id",
    "w1_age_round",
    "w1_tmtb_norm",
    "w1_sf_norm",
    "w1_dsb_norm",
    "w2_tmtb_norm",
    "w2_sf_norm",
    "w2_dsb_norm",
    "w1_superager_ravlt",
    "w2_superager_ravlt",
    "superager_tp1",
    "superager_tp2",
    "superager_long",
]
sample_df[relevant_columns]

In [ ]:
# Create a new df
clean_df = data.copy()

tp1_required_cols = ["w1_superager_ravlt", "w1_tmtb_norm", "w1_sf_norm", "w1_dsb_norm"]
tp2_required_cols = ["w2_superager_ravlt", "w2_tmtb_norm", "w2_sf_norm", "w2_dsb_norm"]
tp1_mask = clean_df[tp1_required_cols].notna().all(axis=1)
tp2_mask = clean_df[tp2_required_cols].notna().all(axis=1)
longitudinal_mask = tp1_mask & tp2_mask

row_count_tp1 = int(tp1_mask.sum())
row_count_tp2 = int(tp2_mask.sum())
row_count_long = int(longitudinal_mask.sum())

superager_count_tp1 = int(clean_df.loc[tp1_mask, "superager_tp1"].sum())
age_matched_controls_t1 = row_count_tp1 - superager_count_tp1
superager_count_long = int(clean_df.loc[longitudinal_mask, "superager_long"].sum())
age_matched_controls_long = row_count_long - superager_count_long
superager_count_tp2 = int(clean_df.loc[tp2_mask, "superager_tp2"].sum())
age_matched_controls_t2 = row_count_tp2 - superager_count_tp2


def pct(count: int, denom: int) -> float:
    """Calculate a percentage.

    Args:
        count: Number of participants in the group.
        denom: Total number of participants.

    Returns:
        Percentage, or NaN if the total is 0.
    """
    return (count / denom * 100) if denom else float("nan")


print(" ")
print(f"Participants with longitudinal data: {row_count_long}")
print(f"Number of longitudinal superagers: {superager_count_long:.0f}")
print(f"Number of age-matched controls long: {age_matched_controls_long:.0f}")
print(f"Percentage longitudinal superagers: {pct(superager_count_long, row_count_long):.2f}%")

print(f"Participants with tp1 data: {row_count_tp1}")
print(f"Number of tp1 superagers: {superager_count_tp1:.0f}")
print(f"Number of age-matched controls tp1: {age_matched_controls_t1:.0f}")
print(f"Percentage tp1 superagers: {pct(superager_count_tp1, row_count_tp1):.2f}%")

print(f"Participants with tp2 data: {row_count_tp2}")
print(f"Number of tp2 superagers: {superager_count_tp2:.0f}")
print(f"Number of age-matched controls tp2: {age_matched_controls_t2:.0f}")
print(f"Percentage tp2 superagers: {pct(superager_count_tp2, row_count_tp2):.2f}%")

# Get basic info about each group
group_stats = {
    "Longitudinal superagers": clean_df[longitudinal_mask & (clean_df["superager_long"] == 1)],
    "tp1 superagers": clean_df[tp1_mask & (clean_df["superager_tp1"] == 1)],
    "tp2 superagers": clean_df[tp2_mask & (clean_df["superager_tp2"] == 1)],
}

for label, subset in group_stats.items():
    if len(subset) == 0:
        print(f"{label}: no participants")
        continue
    avg_age = subset["w1_age"].mean()
    std_age = subset["w1_age"].std()
    print(f"{label}: n={len(subset)}, mean age tp1={avg_age:.2f} (SD {std_age:.2f})")

clean_df[["id", "superager_long", "superager_tp1", "superager_tp2", "w1_age", "YoE"]].sample(15)

In [ ]:
# Filter the df to include only the needed variables
clean_df = clean_df[
    [
        "id",
        "w1_age",
        "YoE",
        "sex",
        "w1_delayed_recall_raw",
        "w1_tmt_b_raw",
        "w1_sem_fluency_raw",
        "w1_inverse_digits_raw",
        "w2_age",
        "w2_delayed_recall_raw",
        "w2_sem_fluency_raw",
        "w2_tmt_b_raw",
        "w2_inverse_digits_raw",
        "w1_ravlt_total",
        "w2_ravlt_total",
        "superager_long",
        "superager_tp1",
        "superager_tp2",
    ]
]

# Export this df to a csv to use for future analysis
clean_df.to_csv(EXPORT_DIR / "superager.csv", index=False)